In [1]:
import glob
import numpy as np
import timm
import torch
from PIL import Image

import matplotlib.pyplot as plt

In [2]:
"""image processing."""

import cv2
import numpy as np


# def convert_to_8_tile(img):
#     img1 = np.expand_dims(img[:500, :500], axis=0)
#     img2 = np.expand_dims(img[:500, 500:1000], axis=0)
#     img3 = np.expand_dims(img[:500, 1000:1500], axis=0)
#     img4 = np.expand_dims(img[:500, 1500:], axis=0)

#     img5 = np.expand_dims(img[500:, :500], axis=0)
#     img6 = np.expand_dims(img[500:, 500:1000], axis=0)
#     img7 = np.expand_dims(img[500:, 1000:1500], axis=0)
#     img8 = np.expand_dims(img[500:, 1500:], axis=0)

#     img_resized = np.expand_dims(cv2.resize(img, (500, 500)), axis=0)

#     img_tiles = np.concatenate(
#         [img_resized, img1, img2, img3, img4, img5, img6, img7, img7, img8], axis=0
#     )
#     print(img_tiles.shape)

#     return img_tiles

def convert_to_8_tile(img):
    img1 = img.crop((0, 0, 500, 500))
    img2 = img.crop((500, 0, 1000, 500))
    img3 = img.crop((1000, 0, 1500,500))
    img4 = img.crop((1500, 0, 2000, 500))

    img5 = img.crop((0, 500, 500, 1000))
    img6 = img.crop((500, 500, 1000, 1000))
    img7 = img.crop((1000, 500, 1500, 1000))
    img8 = img.crop((1500, 500, 2000, 1000))

    img_resized = img.resize((500, 500))

    img_tiles = [img_resized, img1, img2, img3, img4, img5, img6, img7, img7, img8]
    

    return img_tiles


In [3]:
data_config = timm.data.resolve_model_data_config("convnextv2_huge.fcmae_ft_in22k_in1k_512")
transform = timm.data.create_transform(**data_config, is_training=False)
backbone = timm.create_model(
    model_name="convnextv2_huge.fcmae_ft_in22k_in1k_512",
    pretrained=True,
    drop_rate=0.3,
    num_classes=0,)

In [4]:
backbone.eval()

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 352, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((352,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(352, 352, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=352)
          (norm): LayerNorm((352,), eps=1e-06, elementwise_affine=True)
          (mlp): GlobalResponseNormMlp(
            (fc1): Linear(in_features=352, out_features=1408, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (grn): GlobalResponseNorm()
            (fc2): Linear(in_features=1408, out_features=352, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(352, 352, kernel_size=(7, 7), str

In [5]:
img_paths = glob.glob("../data/raw/competitions/csiro-biomass/train/*")

In [6]:
for path in img_paths:
    break

In [7]:
img = Image.open(path)

In [ ]:
img.crop((500,0,1000,500))

In [ ]:
img_arr = np.array(img)

In [ ]:
img_arr.shape

In [ ]:
# height, width

img1 = img_arr[0:500, 0:500].copy()
img2 = img_arr[0:500, 500:1000].copy()
img3 = img_arr[0:500, 1000:1500].copy()
img4 = img_arr[0:500, 1500:].copy()

img5 = img_arr[500:1000, 0:500].copy()
img6 = img_arr[500:1000, 500:1000].copy()
img7 = img_arr[500:1000, 1000:1500].copy()
img8 = img_arr[500:1000, 1500:].copy()

In [ ]:
Image.fromarray(img1)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes[0, 0].imshow(img1)
axes[0, 0].axis('off')
axes[0, 1].imshow(img2)
axes[0, 1].axis('off')
axes[0, 2].imshow(img3)
axes[0, 2].axis('off')
axes[0, 3].imshow(img4)
axes[0, 3].axis('off')

axes[1, 0].imshow(img5)
axes[1, 0].axis('off')
axes[1, 1].imshow(img6)
axes[1, 1].axis('off')
axes[1, 2].imshow(img7)
axes[1, 2].axis('off')
axes[1, 3].imshow(img8)
axes[1, 3].axis('off')
fig.tight_layout()

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(img_arr)

---

In [8]:
img_tiles = convert_to_8_tile(img)

In [9]:
emb_tiles = []

In [ ]:
for tile in img_tiles:
    img_trf = transform(img_tiles[0])
    img_trf = img_trf.unsqueeze(0)
    img_emb = backbone(img_trf)
    emb_tiles.append(img_emb)

In [10]:
img_trf = transform(img_tiles[0])
img_trf = img_trf.unsqueeze(0)
img_emb = backbone(img_trf)
emb_tiles.append(img_emb)

In [11]:
img_trf = transform(img_tiles[1])
img_trf = img_trf.unsqueeze(0)
img_emb = backbone(img_trf)
emb_tiles.append(img_emb)

In [12]:
emb_tiles = torch.cat(emb_tiles, dim=0)

In [13]:
emb_tiles.shape

torch.Size([2, 2816])

In [16]:
emb_tiles.mean(dim=0).shape

torch.Size([2816])

In [18]:
backbone.num_features

2816